# 06 — Generate Transformer Training Data

Generate labeled training data from two sources:
1. **Heston** — Monte Carlo simulation of SPY price paths
2. **GAN** — Inverse-transform synthetic sequences back to price paths

Labels are always discounted expected payoffs: `exp(-rT) * mean(max(S_T - K, 0))`.

**Outputs:** Training datasets for Experiments A (Heston), B (GAN), C (mixed)

In [ ]:
import sys, os
import numpy as np
import pickle
from tqdm import tqdm
sys.path.insert(0, os.path.abspath('..'))

from src.models.heston import simulate_heston, HESTON_DEFAULTS

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
# Load real training sequences (for pairing with simulated labels)
train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
print(f'Real training sequences: {train_seqs.shape}')

# Load scaler for GAN inverse transform
with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

# Load GAN synthetic sequences
gan_seqs_path = os.path.join(PROCESSED_DIR, 'gan_sequences.npy')
if os.path.exists(gan_seqs_path):
    gan_seqs = np.load(gan_seqs_path)
    print(f'GAN sequences: {gan_seqs.shape}')
else:
    print('GAN sequences not found. Run 04_train_gan first.')
    gan_seqs = None

## 1. Generate Heston Training Data (50,000 samples)

In [ ]:
N_HESTON = 50000
rng = np.random.RandomState(42)

# Sample contract parameters from realistic ranges
S0_samples = rng.uniform(250, 500, N_HESTON)     # SPY price range 2010-2023
moneyness_samples = rng.uniform(0.85, 1.15, N_HESTON)
K_samples = S0_samples / moneyness_samples
T_samples = rng.uniform(7/365, 90/365, N_HESTON)  # 7 to 90 days
r_samples = rng.uniform(0.001, 0.055, N_HESTON)   # treasury range

heston_labels = np.zeros(N_HESTON)
heston_contracts = np.zeros((N_HESTON, 4))  # [strike, T, r, moneyness]

print(f'Generating {N_HESTON} Heston prices (n_paths=10000 each)...')
print('This will take a while. Consider reducing n_paths for faster iteration.')

for i in tqdm(range(N_HESTON)):
    price = simulate_heston(
        S0=S0_samples[i], K=K_samples[i], T=T_samples[i], r=r_samples[i],
        n_paths=10000, n_steps=int(T_samples[i] * 252),
        seed=42 + i,
    )
    heston_labels[i] = price
    heston_contracts[i] = [K_samples[i], T_samples[i], r_samples[i], moneyness_samples[i]]

print(f'Heston labels — mean: {heston_labels.mean():.4f}, std: {heston_labels.std():.4f}')
print(f'Non-zero labels: {(heston_labels > 0).sum()} / {N_HESTON}')

In [ ]:
# Save Heston data
np.save(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'), heston_contracts)
np.save(os.path.join(PROCESSED_DIR, 'heston_labels.npy'), heston_labels)
print('Saved Heston training data.')

## 2. Generate GAN Training Data (50,000 samples)

Invert normalization on the log_return feature, reconstruct price paths, then compute option payoffs.

In [ ]:
if gan_seqs is not None:
    N_GAN = 50000
    rng_gan = np.random.RandomState(123)
    
    # Sample contract parameters (same ranges as Heston)
    S0_gan = rng_gan.uniform(250, 500, N_GAN)
    moneyness_gan = rng_gan.uniform(0.85, 1.15, N_GAN)
    K_gan = S0_gan / moneyness_gan
    T_gan = rng_gan.uniform(7/365, 90/365, N_GAN)
    r_gan = rng_gan.uniform(0.001, 0.055, N_GAN)
    
    # Inverse-transform log_return feature (column 0)
    # The scaler was fit on all 20 features; we need mean and std for feature 0
    log_return_mean = scaler.mean_[0]
    log_return_std = scaler.scale_[0]
    
    gan_labels = np.zeros(N_GAN)
    gan_contracts = np.zeros((N_GAN, 4))
    
    print(f'Computing GAN-based option prices for {N_GAN} contracts...')
    
    # Use subsets of GAN sequences as Monte Carlo paths
    n_paths_per_sample = 100  # Each GAN sequence is one path
    
    for i in tqdm(range(N_GAN)):
        # Pick a random batch of GAN sequences
        path_indices = rng_gan.choice(len(gan_seqs), n_paths_per_sample, replace=False)
        batch = gan_seqs[path_indices]  # (n_paths, 60, 20)
        
        # Extract and inverse-transform log returns
        log_returns_scaled = batch[:, :, 0]  # (n_paths, 60)
        log_returns_raw = log_returns_scaled * log_return_std + log_return_mean
        
        # Reconstruct price paths from log returns
        # How many steps to use depends on time to expiry
        n_steps = max(1, int(T_gan[i] * 252))
        n_steps = min(n_steps, 60)  # cap at sequence length
        
        cumulative_returns = np.cumsum(log_returns_raw[:, :n_steps], axis=1)
        S_paths = S0_gan[i] * np.exp(cumulative_returns)
        S_T = S_paths[:, -1]
        
        # Option payoff
        payoffs = np.maximum(S_T - K_gan[i], 0)
        gan_labels[i] = np.exp(-r_gan[i] * T_gan[i]) * np.mean(payoffs)
        gan_contracts[i] = [K_gan[i], T_gan[i], r_gan[i], moneyness_gan[i]]
    
    print(f'GAN labels — mean: {gan_labels.mean():.4f}, std: {gan_labels.std():.4f}')
    print(f'Non-zero labels: {(gan_labels > 0).sum()} / {N_GAN}')
    
    np.save(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'), gan_contracts)
    np.save(os.path.join(PROCESSED_DIR, 'gan_labels.npy'), gan_labels)
    print('Saved GAN training data.')
else:
    print('Skipping GAN data generation (no GAN sequences available).')

## 3. Build Training Datasets for Each Experiment

In [ ]:
from src.data.dataset import SimulatedPricingDataset

# Load Heston data
h_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
h_labels = np.load(os.path.join(PROCESSED_DIR, 'heston_labels.npy'))

# Experiment A: 50,000 Heston paths only
ds_a = SimulatedPricingDataset(train_seqs, h_contracts, h_labels)
print(f'Experiment A (Heston): {len(ds_a)} samples')

if os.path.exists(os.path.join(PROCESSED_DIR, 'gan_contracts.npy')):
    g_contracts = np.load(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'))
    g_labels = np.load(os.path.join(PROCESSED_DIR, 'gan_labels.npy'))
    
    # Experiment B: 50,000 GAN paths only
    ds_b = SimulatedPricingDataset(train_seqs, g_contracts, g_labels)
    print(f'Experiment B (GAN): {len(ds_b)} samples')
    
    # Experiment C: 25,000 Heston + 25,000 GAN mixed
    mixed_contracts = np.concatenate([h_contracts[:25000], g_contracts[:25000]], axis=0)
    mixed_labels = np.concatenate([h_labels[:25000], g_labels[:25000]], axis=0)
    ds_c = SimulatedPricingDataset(train_seqs, mixed_contracts, mixed_labels)
    print(f'Experiment C (Mixed): {len(ds_c)} samples')
else:
    print('GAN data not available. Only Experiment A will be runnable.')

## Quick Sanity Check

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(h_labels, bins=100, alpha=0.7, label='Heston')
if os.path.exists(os.path.join(PROCESSED_DIR, 'gan_labels.npy')):
    axes[0].hist(g_labels, bins=100, alpha=0.7, label='GAN')
axes[0].set_title('Distribution of Training Labels (Option Prices)')
axes[0].set_xlabel('Price ($)')
axes[0].legend()

# Check one sample
sample_input, sample_label = ds_a[0]
axes[1].imshow(sample_input.numpy().T, aspect='auto', cmap='RdBu_r')
axes[1].set_title(f'Sample Input (60x24), label=${sample_label.item():.4f}')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Feature')

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'training_data_overview.png'), dpi=150, bbox_inches='tight')
plt.show()